# Notebook 2 - Advanced: Analyzing the John Snow Cholera Outbreak Using OpenStreetMaps and Networkx - Part 2

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/PHI-Case-Studies/Jupyterlite-1854-Cholera-Outbreak-London-Advanced-1/blob/8bde848d13d71602b2b32c62e5d4b4957e4a45a1/content/NB02-Cholera-Case-Study-OpenStreetMaps-Networkx.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Follow notebook order to regenerate omitted outputs; optional unlicensed media are external continuations.

**Provenance:** `content/NB02-Cholera-Case-Study-OpenStreetMaps-Networkx.ipynb` at `8bde848d13d71602b2b32c62e5d4b4957e4a45a1`; original SHA-256 `255837e40225a1ae33b02d2f0beeb300041f50ed5bf9994e5b49f02aa82cd096`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


In [ ]:
%run ./resources/library.py

# Notebook 2 - Advanced: Analyzing the John Snow Cholera Outbreak Using OpenStreetMaps and Networkx - Part 2

## Learning Objectives

By the end of this notebook, you should be able to:

1. Recall how to load and save `osmnx` street network graphs 
2. Recall how to load data from CSV and pickle files using `pandas`
3. Describe how the three distances, d1, d2 and d3, are computed using `osmnx`
4. Describe how a nested loop works to calculate the route distance between 2 nodes (pumps and deaths)
5. Identify how this algorithm pattern can be used in solving other public health problems.

## Summary of steps

We will carry out the following <font color=red>steps</font>:

<font color=red>Step 1</font> - Read the street network graph, `G`, of Soho district using OSMnx using a set of coordinates in the middle of Soho district. **(We saved this graph in Notebook 2 and we will read it from the graphml file `soho.graphml`.)**  
   
   
<font color=red>Step 2</font> - Load the original data sets from Notebook 1, Basic, (pumps and deaths) into `pumps_df` and `deaths_df`.


<font color=red>Step 3</font> - To represent coordinates from the pumps and deaths from the Notebook 1 in OSMnx graph format, we have to find the nearest OSMnx nodes to those points. We will add new columns to the pumps and deaths dataframes to accomodate new information coming from OSMnx. We will also store the short distances between original points to the nearest OSMnx points and store it in the respective dataframes.

<font color=red>Step 4</font> -  To calculate mean distances from death coordinates to pump coordinates we will create a nested loop through records of both dataframes, `pumps_df` and `deaths_df`, for pairwise distance calculations between each pump and death coordinates. We will add the short distances from #1 to the pump point to death point distance and store this in a new dataframe called `routes_df`.

<font color=red>Step 5</font> - We will then create the map representation pump-to-death-points mean distances using `folium` and superimpose this on the markers generated in Notebook 1.

In [ ]:
import osmnx as ox
# some configuration parameters for osmnx
# set log to True to for debugging

ox.config(
    log_console=True, 
    use_cache=True, 
    log_file=True,
    overpass_endpoint='https://overpass-api.de/api/interpreter',
    overpass_rate_limit=True,
    timeout=240
)

ox.__version__

Let's read our street network graph from file using the dot function, <font color=red>`load_graphml()`</font>.

In [ ]:
G = ox.load_graphml(filepath='outputs/soho.graphml')

# Project to British National Grid (EPSG:27700)
G_proj = ox.project_graph(G, to_crs='EPSG:27700')

print(G.graph['crs'])
print(G_proj.graph['crs'])



In [ ]:
fig, ax = ox.plot_graph(G_proj, figsize=(14,14), \
                        bgcolor='white', \
                        node_color='teal')

## Step 2. Load pumps and deaths data sets

Let's read the data set from a pickle file using the dot function <font color=red>`read_pickle()`</font>. 

In [ ]:
import pandas as pd

# Read pandas pickle file for cholera deaths.
deaths_df = pd.read_pickle('outputs/mean_center_df.pickle')

# display dataframe records
deaths_df

In [ ]:
import geopandas

gdf = geopandas.GeoDataFrame(
    deaths_df, geometry=geopandas.points_from_xy(deaths_df.LON, deaths_df.LAT),\
    crs='epsg:4236')

This dataframe has two additional columns, `product_LAT` and `product_LON` resulting from computing for the mean center in Notebook 2.

Next, let's read the pickle file for pumps.

In [ ]:
# Read CSV with pumps.
pumps_df = pd.read_pickle('outputs/pumps_df.pickle')

pumps_df

## Step 3: Set up `pumps_df` and `deaths_df` dataframes to store additional `osmnx` information

### Set up `pumps_df` dataframe for analysis

Let's create five new columns for `pumps_df`:
1. Two (2),<font color=red>`pumps_df['GLON']`</font> and <font color=red>`pumps_df['GLAT']`</font>, to store coordinates from the OSMnx graph nodes that correspond to original pump coordinates.
2. <font color=red>`pumps_df['DISTANCE']`</font> to store distance between original pump coordinates and OSMnx graph coordinates
3. <font color=red>`pumps_df['NODE']`</font> to store the node ID of an OSMnx graph node coordinate 
4. <font color=red>`pumps_df['MEAN_DISTANCE']`</font> to store the mean distance values between pump and death graph coordinates (to be computed later).

We also store default values for these columns as below.

In [ ]:
pumps_df['GLON'] = 0.00
pumps_df['GLAT'] = 0.00
pumps_df['NEAREST-NODE-DISTANCE'] = 0.00
pumps_df['NODE'] = 0
pumps_df['NODE'].astype(int)
pumps_df['DEATHS-MEAN-DISTANCE'] = 0.00

pumps_df

You can verify the data types for `pumps_df` columns with <font color=red>`dtypes`</font> dataframe attribute like so.

In [ ]:
pumps_df.dtypes

We use a cell magic <font color=red>`%%time`</font> to time the execution of each loop.

In [ ]:
import osmnx as ox

In [ ]:
from shapely.geometry import Point
from shapely.ops import transform
import pyproj
 
project = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True).transform
 
for idx, row in pumps_df.iterrows():
    # Convert to projected coordinates
    point_proj = transform(project, Point(row['LON'], row['LAT']))
    x_proj, y_proj = point_proj.x, point_proj.y
 
    # Get nearest node and distance in meters
    pnode, pdistance = ox.distance.nearest_nodes(G_proj, X=x_proj, Y=y_proj, return_dist=True)
 
    # Assign results
    pumps_df.at[idx, 'GLON'] = G_proj.nodes[pnode]['x']
    pumps_df.at[idx, 'GLAT'] = G_proj.nodes[pnode]['y']
    pumps_df.at[idx, 'NEAREST-NODE-DISTANCE'] = pdistance
    pumps_df.at[idx, 'NODE'] = pnode


The code above obtains the distance between pump coordinates and OSMnx node coordinates using the <font color=red>`ox.get_nearest_node()`</font> as shown in the diagram below.

**Figure 1.** In computing for node-to-node distance using `osmnx`, we have to account for the small distance between the actual lat/lon point representing either pumps or death locations and `osmnx` nodes. This diagram shows the approach for calculating this short distance using `osmnx` dot function called `ox.get_nearest_node()` - EQUATION 1 below.

[Source image continuation: ](https://github.com/PHI-Case-Studies/Jupyterlite-1854-Cholera-Outbreak-London-Advanced-1/blob/8bde848d13d71602b2b32c62e5d4b4957e4a45a1/content/images/osmnx1.png)

To quickly obtain the mean of all values from the `NEAREST-NODE-DISTANCE` column, we use the `pandas` dot function, <font color=red>`mean()`</font>.

In [ ]:
pumps_df['NEAREST-NODE-DISTANCE'].mean()

You can make the print out more human-friendly by adding "meters".

In [ ]:
print(pumps_df['NEAREST-NODE-DISTANCE'].mean(), "meters")

> What does this value mean?

In [ ]:
pumps_df['NEAREST-NODE-DISTANCE'].max()

In [ ]:
pumps_df['NEAREST-NODE-DISTANCE'].min()

In [ ]:
pumps_df

In [ ]:
ax = pumps_df.plot.bar(y='NEAREST-NODE-DISTANCE', x='Street', color='k')
ax.set_title('Distance from Pump to Nearest OSMNx Node')
ax.set_ylabel('Distance, meters')
ax.set_xlabel('Pumps')

The `NEAREST-NODE-DISTANCE` column in `pumps_df` represent `d1` in Figure 1.

### Set up `deaths_df` dataframe for analysis

In [ ]:
deaths_df = deaths_df[['DEATHS', 'LON', 'LAT']]
deaths_df['DEATHS'].astype(int)
deaths_df['GLON'] = 0.00
deaths_df['GLAT'] = 0.00
deaths_df['NEAREST-NODE-DISTANCE'] = 0.00
deaths_df['NODE'] = 0
deaths_df['NODE'].astype(int)
deaths_df.dtypes

In [ ]:
deaths_df.tail()

In [ ]:
%%time
from shapely.geometry import Point
from shapely.ops import transform
import pyproj


project = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True).transform
 
for idx, row in deaths_df.iterrows():
    # Convert to projected coordinates
    point_proj = transform(project, Point(row['LON'], row['LAT']))
    x_proj, y_proj = point_proj.x, point_proj.y
 
    # Get nearest node and distance in meters
    pnode, pdistance = ox.distance.nearest_nodes(G_proj, X=x_proj, Y=y_proj, return_dist=True)
 
    # Assign results
    deaths_df.at[idx, 'GLON'] = G_proj.nodes[pnode]['x']
    deaths_df.at[idx, 'GLAT'] = G_proj.nodes[pnode]['y']
    deaths_df.at[idx, 'NEAREST-NODE-DISTANCE'] = pdistance
    deaths_df.at[idx, 'NODE'] = pnode

In [ ]:
deaths_df['NEAREST-NODE-DISTANCE'].mean()

In [ ]:
deaths_df.head()

In [ ]:
deaths_df.dtypes

In [ ]:
deaths_df['NEAREST-NODE-DISTANCE'].mean()

In [ ]:
deaths_df['NEAREST-NODE-DISTANCE'].min()

In [ ]:
deaths_df['NEAREST-NODE-DISTANCE'].max()

In [ ]:
ax = deaths_df['NEAREST-NODE-DISTANCE'].plot.hist(bins=30, color='k')
ax.set_title('Frequency Distribution, Distance from \nMortality Locations to Nearest OSMNx Node')
ax.set_ylabel('Frequency')
ax.set_xlabel('Distance from Mortality Locations to \nNearest OSMNx Node, meters')

## Step 4: Pairwise calculation of distance between `deaths_df` and `pumps_df` coordinates using a nested loop

### Explanation of algorithm for pair-wise calculations

This code snippet below obtains the total distance, `d1 + d2 + d3`.
```
    d1 = row_pump['NEAREST-NODE-DISTANCE']
    d2 = row_death['NEAREST-NODE-DISTANCE']
    d3 = nx.shortest_path_length(G, death_node, pump_node, weight='length')
    total_distance = d1 + d2 + d3

    distance = \
          nx.shortest_path_length(G, death_node, pump_node, weight='length') + \ 
          row_death[1]['DISTANCE'] + row_pump[1]['DISTANCE']
```
By this time we have obtained three sets of distance values, `d1`, `d2` and `d3` (see Figure 2 below):
1. <font color=red>`d1`</font> = distance in meters between coordinates of an actual pump and an OSMnx node from graph, `G` from `row_death[1]['DISTANCE']`
2. <font color=red>`d2`</font> = distance in meters between coordinates of a death location and an OSMnx node from graph, `G` from `row_pump[1]['DISTANCE']`
3. <font color=red>`d3`</font> = distance in meters between coordinates of two OSMnx nodes (pump and death location, or `PNODE` and `DNODE`, respectively in the `routes_df` dataframe) from <font color=red>`nx.shortest_path_length(G, dnode, pnode, weight='length')`</font>

**Figure 2.** Diagram describing the measurement of node to node distance.

[Source image continuation: ](https://github.com/PHI-Case-Studies/Jupyterlite-1854-Cholera-Outbreak-London-Advanced-1/blob/8bde848d13d71602b2b32c62e5d4b4957e4a45a1/content/images/osmnx2.png)

In [ ]:
import networkx as nx
# create empty list, lst, to store node and distance data 
#    from pairwise matching
new_list = []

# carry out pairwise calculations
# outer loop
for index, row_pump in pumps_df.iterrows():
    # inner loop
    for index, row_death in deaths_df.iterrows():
        death_node = row_death['NODE']
        pump_node = row_pump['NODE']
        d1 = row_pump['NEAREST-NODE-DISTANCE']
        d2 = row_death['NEAREST-NODE-DISTANCE']
        d3 = nx.shortest_path_length(G, death_node, pump_node, weight='length') 
        distance = d1 + d2 + d3
        new_list.append([int(death_node), int(pump_node), distance])     

Let's create a new dataframe, `routes_df` from the list.

In [ ]:
columns = ['DNODE','PNODE','PD_DISTANCE']
routes_df = pd.DataFrame(new_list, columns=columns)

In [ ]:
routes_df

In [ ]:
routes_df['PD_DISTANCE'].min()

In [ ]:
routes_df['PD_DISTANCE'].max()

In [ ]:
ax = routes_df['PD_DISTANCE'].plot.hist(bins=30, color='k')
ax.set_title('Frequency Distribution, Total Distance from \nMortality Locations to Pumps')
ax.set_ylabel('Frequency')
ax.set_xlabel('Total Distance, meters')

### Walkability Assumption, 400 meters

How far would Soho district residents walk to get water?

Using the last histogram above, let's assume that the residents of Soho district would be willing to walk 400 meters to fetch water from each pump by setting a filter of 400 meters or less as "walkable". Let's create a new dataframe, `routes400m_df`, containing node-to-node distance values <= 400 meters.

In [ ]:
routes400m_df = routes_df[routes_df['PD_DISTANCE']<=400]

Let's inspect that filtered dataframe, `routes400m_df`. 

After filtering, how many rows did we end up with? What's the shortest / longest walk in this new dataset of pump-death inter-node distance?

In [ ]:
routes400m_df

In [ ]:
routes400m_df.info()

In [ ]:
ax = routes400m_df['PD_DISTANCE'].plot.hist(bins=30, color='k')
ax.set_title('Frequency Distribution, Distance from \nMortality Locations to Pumps, <=400m')
ax.set_ylabel('Frequency')
ax.set_xlabel('Distance from mortality locations to pumps, meters')

We can compare `routes_df` and `routes400m_df` for number of records easily using the `print()` statement below.

In [ ]:
print('routes_df record count (unfiltered): ', len(routes_df), "\n",\
      'routes400m_df record count (filtered, 400 meters): ', len(routes400m_df))

### Under the hood: Visualizing randomly sampled node-to-node distances from computations above

Let's see what's happening under the hood by visualizing some randomly sampled routes between death and pump points. Let's look at 20 samples of death node-pump node pairs. We will use `random.choice` from the python package, `numpy` to select three random death nodes. To keep the pump node constant for each three samples, we will place the random pump code outside the loop. Let's pick the pump node representing the pump at Broadwick Street.

In [ ]:
#%%time
import numpy as np
import matplotlib.pyplot as plt

# Pump node at Broadwick Street
pump_node = 25473293

pump_lat = G.nodes[pump_node]['y']
pump_lon = G.nodes[pump_node]['x']
route_list = []
route_color = 'red'
node_color = 'teal'
bgcolor = 'white'
number_of_node_samples = 20

for i in range(number_of_node_samples):
    death_node = np.random.choice(routes_df['DNODE'].unique())
    route = nx.shortest_path(G, death_node, pump_node)
    route_list.append(route)

fig, ax = ox.plot_graph_routes(G, route_list, node_alpha=0.5, edge_alpha=0.5, \
                            bgcolor=bgcolor, node_color=node_color,\
                            edge_linewidth=1,\
                            route_linewidth=2,\
                            route_color=route_color, \
                            route_alpha=0.3, show=False, close=False, \
                            figsize=(17,17))
ax.scatter(pump_lon, pump_lat, c='red', s=300)
ax.text(pump_lon-0.0002, pump_lat+0.0002, 'Pump', fontsize=12)
ax.set_title('Walking Routes <=400m to Broad Street Pump from 20 Random Mortality Locations', pad=1, size=20)

plt.show()

In [ ]:
routes400m_df.to_pickle('outputs/routes400m_df.pickle')

You can re-run the code above to see more random pairs.

### Updating `pumps_df` with mean distance values from `d1 + d2 + d3` calculations

Let's store the mean distances between each pump and death locations from `routes_df` to `pumps_df`. 

In [ ]:
pumps_df

In [ ]:
for index, row in pumps_df.iterrows():
    print(row)
    pump_death_distance_400m = routes400m_df.loc[(routes400m_df['PNODE'] == row['NODE'])].PD_DISTANCE
    pump_death_distance = routes_df.loc[(routes_df['PNODE'] == row['NODE'])].PD_DISTANCE
    mean_distance_400m = \
        pump_death_distance_400m.mean()
    mean_distance = \
        pump_death_distance.mean()
    print('DISTANCE LIST 400m:', pump_death_distance_400m.to_list())
    print('=========MEAN DISTANCE 400m:', mean_distance_400m)
    print('DISTANCE LIST:', pump_death_distance.to_list())
    print('=========MEAN DISTANCE:', mean_distance_400m)
    pumps_df.loc[index, 'MEAN-PD-DISTANCE-400M'] = mean_distance_400m
    pumps_df.loc[index, 'MEAN-PD-DISTANCE'] = mean_distance
    #print("Pump "+str(row[0])+":\n ","Node ID: ", row[1]['NODE']\
    #      ,"\n","Mean Distance: ", mean_distance, "meters")


In [ ]:
pumps_df

> What do the columns `MEAN-PD-DISTANCE` and `MEAN-PD-DISTANCE-400M` mean?

In [ ]:
ax = pumps_df.plot.bar(y='MEAN-PD-DISTANCE', x='Street',color='k')
ax.set_title('Mean Distance from Mortality Locations (<=400m) to Pumps')
ax.set_ylabel('Mean distance, meters')
ax.set_xlabel('Street Pump')

In [ ]:
ax = pumps_df.plot.bar(y='MEAN-PD-DISTANCE-400M', x='Street',color='k')
ax.set_title('Mean Distance from Mortality Locations (<=400m) to Pumps')
ax.set_ylabel('Mean distance, meters')
ax.set_xlabel('Street Pump')

1. What do the street pumps having no bars mean? 
2. Which street pump has the shortest mean distance from mortality locations (less than 400 m walk)?

## Step 5: Create `folium` map to show which pumps have mean shortest walkable distance values to most death points

### Recreate Notebook 1 `folium` map and markers for pumps and deaths

In [ ]:
mean_center_df = pd.read_pickle('outputs/mean_center_df.pickle')
mean_LON = np.sum(mean_center_df['product_LON'])/np.sum(mean_center_df['DEATHS'])
mean_LAT = np.sum(mean_center_df['product_LAT'])/np.sum(mean_center_df['DEATHS'])
mean_center_df

## Representing the `PD-MEAN-DISTANCE` as circle markers on Folium

In [ ]:
import pandas as pd
import folium

# let's import the folium plugins
from folium import plugins

# we already have pumps_df and deaths_df

SOHO_COORDINATES = (51.513578, -0.136722)

map2 = folium.Map(location=SOHO_COORDINATES, zoom_start=16.5, tiles='cartodbpositron')

locationlist = deaths_df[["LAT","LON"]].values.tolist()
radiuslist = deaths_df[["DEATHS"]].values.tolist()

# Iterate through the rows of the cholera deaths data frame 
#    and add each cholera death circle marker to map1
for i in range(0, len(locationlist)):

    # Create a popup for each marker
    # Each marker will show point information and 
    #   number of deaths in that location.    
    popup = folium.Popup('Location: '+'('+str(locationlist[i][0])+\
                         ', '+str(locationlist[i][1])+')'+\
                         '<br/>'+\
                        'Deaths: '+ str(radiuslist[i][0]))
    
    # Add each circle marker with popup representing 
    #   a location with deaths to map1
    folium.CircleMarker(
                    location=locationlist[i], \
                    radius=radiuslist[i], \
                    popup=popup, \
                    color='black', \
                    weight=1, \
                    fill=True, \
                    fill_color='red', \
                    fill_opacity=1).add_to(map2)

# Iterate through the rows of the pumps_df data frame 
#   and add each water pump to map1
for each in pumps_df.iterrows():
    
    # Create a popup for each marker
    # Each marker will show pump location information
    popup = folium.Popup('Location: '+'('+str(each[1]['LAT'])+', '+str(each[1]['LON'])+')')

    # Add each circle marker with popup representing 
    #   a pump location to map1
    pump_marker = folium.RegularPolygonMarker([each[1]['LAT'],each[1]['LON']], \
                    color='black', \
                    weight=1,\
                    fill_opacity=1, \
                    fill_color='blue', \
                    number_of_sides=4, \
                    popup=popup, \
                    radius=10).add_to(map2)

mean_center_df = pd.read_pickle('outputs/mean_center_df.pickle')
mean_LON = np.sum(mean_center_df['product_LON'])/np.sum(mean_center_df['DEATHS'])
mean_LAT = np.sum(mean_center_df['product_LAT'])/np.sum(mean_center_df['DEATHS'])
mean_center_POINT = (mean_LAT, mean_LON)

folium.CircleMarker(
            location=mean_center_POINT, \
            color='black', \
            weight=2, \
            fill_opacity=1, \
            fill_color="yellowgreen", \
            popup=folium.Popup('Mean Center Point: '+ \
                str(mean_center_POINT)), \
            radius=10).add_to(map2)

for row in pumps_df.iterrows():
    # add each water pump mean distance to map1 (divide by 5 to make circle smaller)
    folium.Circle(location=[row[1]['LAT'],row[1]['LON']], \
                  popup='Pump '+str(row[1]['NODE']), \
                  radius=(row[1]['MEAN-PD-DISTANCE'])/5).add_to(map2)


# let's use the "Fullscreen" plugin
# add the button to the top right corner
plugins.Fullscreen(
    position='topright',
    title='Expand me',
    title_cancel='Exit me',
    force_separate_button=True
).add_to(map2)

map2

Please do not forget to "amplify cognition" by studying tile settings.

## Congratulations!

You have just gained some expertise in:
1. The Cholera Outbreak in 1854 London
2. Use of a few Python packages for data analysis and visualization: `pandas`, `folium`
3. Use of the Open Street Maps - NetworkX package, `osmnx` for street network type analysis
4. Generating value out of information sources (mortality and street network data)

## References

1. Boeing, Geoff. OSMnx: Python for Street Networks. URL: https://geoffboeing.com/2016/11/osmnx-python-street-networks/
2. Networkx. URL: https://networkx.github.io/
3. Shiode S. Revisiting John Snow's map: network-based spatial demarcation of cholera area. International Journal of Geographical Information Science Volume 26, 2012 - Issue 1. URL: https://www.tandfonline.com/doi/abs/10.1080/13658816.2011.577433.

*For case study suggestions for improvement, please contact Herman Tolentino, Jan MacGregor, James Tobias or Zhanar Haimovich.*